In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Extract ZIP file
from pathlib import Path
import zipfile

zip_path = Path("/content/drive/MyDrive/MMA3001_v2_folder/dataset/pork_rasher_v2.zip")
extract_path = Path("/content/pork_rasher_dataset_v2")

if not zip_path.exists():
    print("Error: ZIP file not found at the specified path.")
    print("Please check if the file exists and if Google Drive is mounted correctly.")
else:
    extract_path.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    print("Dataset extracted successfully!")

    # Display the extracted folder structure
    print("\nExtracted files and folders:")
    for item in sorted(extract_path.iterdir()):
        print(item.name)

Dataset extracted successfully!

Extracted files and folders:
README.dataset.txt
README.roboflow.txt
data.yaml
test
train
valid


In [3]:
# 2. read data.yaml, confirmation for 4 defect classes
import yaml

# locate the dataset configuration
dataset_path = Path("/content/pork_rasher_dataset_v2")
yaml_path = dataset_path / "data.yaml"

# read the configuration
with open(yaml_path, "r") as file:
  config = yaml.safe_load(file)

# Display the configuration
print("Dataset configuration:\n")
for key, value in config.items():
  print(f"{key}: {value}")

Dataset configuration:

train: ../train/images
val: ../valid/images
test: ../test/images
nc: 4
names: ['loose-meat', 'packaging-error', 'twisted-meat', 'unsealed']
roboflow: {'workspace': 'hello-2aqe0', 'project': 'pork-rasher-error-packaging', 'version': 2, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/hello-2aqe0/pork-rasher-error-packaging/dataset/2'}


In [4]:
# 3. count images and annotations, so we know if the data split is reasonable
from collections import Counter

dataset_path = Path("/content/pork_rasher_dataset_v2")
class_names = config["names"]

# check each dataset split
for split in ["train", "valid", "test"]:
  image_dir = dataset_path / split / "images"
  label_dir = dataset_path / split / "labels"

  # count images
  images = [
      p for p in image_dir.iterdir()
      if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
  ]

  # count bounding box annotations
  class_counts = Counter()
  for label_path in label_dir.iterdir():
    with open(label_path, "r") as file:
      for line in file:
        if line.strip():
          class_id = int(line.split()[0])
          class_counts[class_id] += 1

  # display results
  print(f"\n======{split.upper()} SET======")
  print("Total images:", len(images))

  for class_id, class_name in enumerate(class_names):
    print(f"{class_name}: {class_counts[class_id]}")


======TRAIN SET======
Total images: 1944
loose-meat: 65
packaging-error: 527
twisted-meat: 67
unsealed: 1048

======VALID SET======
Total images: 120
loose-meat: 3
packaging-error: 34
twisted-meat: 3
unsealed: 81

======TEST SET======
Total images: 80
loose-meat: 3
packaging-error: 23
twisted-meat: 4
unsealed: 49


In [5]:
# 4. Create cleaned dataset to avoid source-image leakage
import shutil

original = dataset_path
clean = Path("/content/pork_rasher_dataset_v2_clean")

# Copy original dataset into a fresh folder
shutil.rmtree(clean, ignore_errors=True)
shutil.copytree(original, clean)

def source_ids(split):
    return {
        p.name.split(".rf.")[0]
        for p in (clean / split / "images").glob("*.jpg")
    }

# Remove overlapping images and their labels
for split, blocked in [
    ("train", source_ids("valid") | source_ids("test")),
    ("valid", source_ids("test"))
]:
    for image in (clean / split / "images").glob("*.jpg"):
        if image.name.split(".rf.")[0] in blocked:
            image.unlink()
            (clean / split / "labels" / f"{image.stem}.txt").unlink()

# Update YOLO dataset configuration
config.update({
    "path": str(clean),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images"
})

(clean / "data.yaml").write_text(yaml.safe_dump(config))

# Confirm no shared source identifiers
assert not (
    source_ids("train") & source_ids("valid")
    or source_ids("train") & source_ids("test")
    or source_ids("valid") & source_ids("test")
)

print("No overlapping source identifiers between dataset splits.")

No overlapping source identifiers between dataset splits.


In [6]:
# 5. Verify cleaned images and annotations

for split in ["train", "valid", "test"]:

    images = list((clean / split / "images").glob("*.jpg"))
    counts = Counter()

    for image in images:
        label = clean / split / "labels" / f"{image.stem}.txt"
        assert label.exists(), f"Missing label: {image.name}"

        for line in label.read_text().splitlines():
            if not line.strip():
                continue

            values = list(map(float, line.split()))
            assert len(values) == 5, f"Invalid label: {label.name}"

            class_id, x, y, w, h = values

            # Verify class ID and bounding-box coordinates
            assert class_id.is_integer() and 0 <= class_id < len(class_names)
            assert w > 0 and h > 0
            assert 0 <= x - w/2 <= x + w/2 <= 1
            assert 0 <= y - h/2 <= y + h/2 <= 1

            counts[int(class_id)] += 1

    # Display final results
    print(f"\n{split.upper()}: {len(images)} images")
    for i, name in enumerate(class_names):
        print(f"{name}: {counts[i]}")

print("\nAnnotation verification complete.")


TRAIN: 1935 images
loose-meat: 65
packaging-error: 521
twisted-meat: 67
unsealed: 1045

VALID: 119 images
loose-meat: 3
packaging-error: 34
twisted-meat: 3
unsealed: 80

TEST: 80 images
loose-meat: 3
packaging-error: 23
twisted-meat: 4
unsealed: 49

Annotation verification complete.


In [7]:
%pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 7.7 MB/s eta 0:00:00


In [8]:
# 6. Load the trained V2 baseline
from ultralytics import YOLO
model_path = Path(
    "/content/drive/MyDrive/MMA3001_v2_folder/model_runs/"
    "yolov8n_v2_baseline/weights/best.pt"
)

model = YOLO(str(model_path))
print("V2 baseline loaded successfully!")

# 7. Evaluate V2 on unseen test images
test_results = model.val(
    data=str(clean / "data.yaml"),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/content/drive/MyDrive/MMA3001_v2_folder/model_runs",
    name="yolov8n_v2_baseline_test"
)

print("V2 test evaluation complete!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
V2 baseline loaded successfully!
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1918.4±912.5 MB/s, size: 66.1 KB)
val: Scanning /content/pork_rasher_dataset_v2_clean/test/labels... 80 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 909.7it/s 0.1s
val: New cache created: /content/pork_rasher_dataset_v2_clean/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.8it/s 2.8s
                   all         80         79      0.775      0.439      0.499